### Imports

In [1]:
import sys
import os
import re
import json
import logging
from datetime import datetime
from bs4 import BeautifulSoup

In [2]:
# Add parent directory to path for imports
sys.path.insert(0, '../../scrapers')  # Go up to scrapers folder
from utils.http_client import HttpClient

### Configuration

In [3]:
# =============================================================================
# CONFIGURATION -- to be changed for testing
# =============================================================================

SOURCE_NAME = "vesselfinder"
OUTPUT_DIR = "data/raw/vesselfinder"

In [4]:
# Chilean ports to monitor
PORTS = {
    "CLSAI001": "San Antonio",
    "CLVAP001": "Valparaíso",
}

BASE_URL = "https://www.vesselfinder.com/ports"

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

### Parser functions

In [5]:
def parse_port_summary(soup):
    """Extract expected ships and ships in port counts."""
    summary = {
        "expected_ships": None,
        "ships_in_port": None
    }
    
    pei_div = soup.find("div", class_="pei")
    if pei_div:
        text = pei_div.get_text()
        
        expected_match = re.search(r"Expected ships:\s*(\d+)", text)
        in_port_match = re.search(r"Ships in port:\s*(\d+)", text)
        
        if expected_match:
            summary["expected_ships"] = int(expected_match.group(1))
        if in_port_match:
            summary["ships_in_port"] = int(in_port_match.group(1))
    
    return summary

In [7]:
def parse_vessel_row(row):
    """Parse a single vessel row from the table."""
    cells = row.find_all("td")
    if len(cells) < 2:
        return None
    
    vessel = {}
    
    # First column: time (ETA, arrival, departure, or last report)
    vessel["time"] = cells[0].get_text(strip=True)
    
    # Second column: vessel info
    vessel_cell = cells[1]
    
    # Vessel name
    named_title = vessel_cell.find("div", class_="named-title")
    if named_title:
        vessel["vessel_name"] = named_title.get_text(strip=True)
    
    # Vessel type
    named_subtitle = vessel_cell.find("div", class_="named-subtitle")
    if named_subtitle:
        vessel["vessel_type"] = named_subtitle.get_text(strip=True)
    
    # IMO from link
    vessel_link = vessel_cell.find("a", class_="named-item")
    if vessel_link and vessel_link.get("href"):
        href = vessel_link.get("href")
        imo_match = re.search(r"/details/(\d+)", href)
        if imo_match:
            vessel["imo"] = imo_match.group(1)
    
    # Flag from image title
    flag_div = vessel_cell.find("div", class_="m-flag-small")
    if flag_div and flag_div.get("title"):
        vessel["flag"] = flag_div.get("title")
    
    # Additional columns
    for cell in cells:
        classes = cell.get("class", [])
        text = cell.get_text(strip=True)
        
        if "col-y" in classes and text and text != "-":
            try:
                vessel["built"] = int(text)
            except ValueError:
                pass
        
        if "col-gt" in classes and text and text != "-":
            try:
                vessel["gt"] = int(text.replace(",", ""))
            except ValueError:
                pass
        
        if "col-dwt" in classes and text and text != "-":
            try:
                vessel["dwt"] = int(text.replace(",", ""))
            except ValueError:
                pass
        
        if "col-sizes" in classes and text and text != "-":
            vessel["size"] = text
    
    return vessel if vessel.get("vessel_name") else None

In [9]:
def parse_vessel_table(soup, table_id):
    """Parse a vessel table (expected, arrivals, departures, in-port)."""
    vessels = []
    
    section = soup.find("section", id=table_id)
    if not section:
        return vessels
    
    table = section.find("table")
    if not table:
        return vessels
    
    tbody = table.find("tbody")
    if not tbody:
        return vessels
    
    for row in tbody.find_all("tr"):
        vessel = parse_vessel_row(row)
        if vessel:
            vessels.append(vessel)
    
    return vessels

### Scraper class

In [10]:
class VesselFinderPortScraper:
    """Scrapes port status from VesselFinder."""
    
    def __init__(self):
        self.client = HttpClient(delay=2)
    
    def scrape_port(self, port_code, port_name):
        """Scrape all data for a single port."""
        url = f"{BASE_URL}/{port_code}"
        html = self.client.fetch(url)
        
        if not html:
            logger.error(f"Failed to fetch {port_name}")
            return None
        
        soup = BeautifulSoup(html, "html.parser")
        
        data = {
            "source": SOURCE_NAME,
            "port_code": port_code.replace("001", ""),  # CLSAI001 -> CLSAI
            "port_name": port_name,
            "scrape_timestamp": datetime.utcnow().isoformat() + "Z",
            "summary": parse_port_summary(soup),
            "expected_vessels": parse_vessel_table(soup, "expected"),
            "vessels_in_port": parse_vessel_table(soup, "in-port"),
            "recent_arrivals": parse_vessel_table(soup, "arrivals"),
            "recent_departures": parse_vessel_table(soup, "departures"),
        }
        
        return data

In [11]:
def scrape_all_ports(self):
        """Scrape all configured ports."""
        all_data = []
        
        for port_code, port_name in PORTS.items():
            logger.info(f"Scraping {port_name} ({port_code})")
            
            try:
                data = self.scrape_port(port_code, port_name)
                if data:
                    all_data.append(data)
                    summary = data["summary"]
                    logger.info(
                        f"  Expected: {summary['expected_ships']}, "
                        f"In Port: {summary['ships_in_port']}"
                    )
            except Exception as e:
                logger.error(f"Failed to scrape {port_name}: {e}")
        
        return all_data

### Output functions

In [12]:
def ensure_output_dir():
    """Create output directory if it doesn't exist."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

In [13]:
def save_data(data):
    """Save scraped data to JSON file."""
    ensure_output_dir()
    
    timestamp = datetime.utcnow().strftime("%Y-%m-%d_%H-%M")
    filename = f"{OUTPUT_DIR}/ports_{timestamp}.json"
    
    output = {
        "scrape_type": "port_status",
        "scrape_timestamp": datetime.utcnow().isoformat() + "Z",
        "ports_scraped": len(data),
        "data": data
    }
    
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    
    logger.info(f"Saved to {filename}")
    return filename

In [14]:
OUTPUT_DIR

'data/raw/vesselfinder'

In [15]:
def print_summary(data):
    """Print summary of scraped data."""
    print("\n" + "=" * 60)
    print("PORT STATUS SUMMARY")
    print("=" * 60)
    
    for port in data:
        summary = port["summary"]
        print(f"\n{port['port_name']} ({port['port_code']})")
        print(f"  Expected ships: {summary['expected_ships']}")
        print(f"  Ships in port:  {summary['ships_in_port']}")
        print(f"  Vessels tracked:")
        print(f"    - Expected:   {len(port['expected_vessels'])}")
        print(f"    - In port:    {len(port['vessels_in_port'])}")
        print(f"    - Arrivals:   {len(port['recent_arrivals'])}")
        print(f"    - Departures: {len(port['recent_departures'])}")

### Main function

In [16]:
scraper = VesselFinderPortScraper()

#### testing scrape_port from Class VesselFinderPortScraper

In [17]:
PORTS

{'CLSAI001': 'San Antonio', 'CLVAP001': 'Valparaíso'}

In [18]:
# testing san antonio
port_code = 'CLSAI001'
port_name = PORTS[port_code]

In [19]:
url = f"{BASE_URL}/{port_code}"

In [20]:
html = scraper.client.fetch(url)

2026-03-10 14:32:44,718 [INFO] Fetching https://www.vesselfinder.com/ports/CLSAI001 (attempt 1/3)


In [21]:
if not html:
    logger.error(f"Failed to fetch {port_name}")
    # return None

soup = BeautifulSoup(html, "html.parser")

In [22]:
SOURCE_NAME, port_code.replace("001", ""), port_name, datetime.utcnow().isoformat() + "Z"

('vesselfinder', 'CLSAI', 'San Antonio', '2026-03-10T13:32:49.126875Z')

In [23]:
parse_port_summary(soup)

{'expected_ships': 11, 'ships_in_port': 18}

In [24]:
parse_vessel_table(soup, "expected"),

([{'time': 'Mar 11, 00:00',
   'vessel_name': 'MSC EDNA',
   'vessel_type': 'Container Ship',
   'imo': '1016680',
   'flag': 'Liberia',
   'built': 2025,
   'gt': 24464,
   'dwt': 40515,
   'size': '183 x 32'},
  {'time': 'Mar 12, 14:05',
   'vessel_name': 'ASL SAKURA',
   'vessel_type': 'Bulk Carrier',
   'imo': '9961922',
   'flag': 'Marshall Islands',
   'built': 2025,
   'gt': 24464,
   'dwt': 40515,
   'size': '183 x 32'},
  {'time': 'Mar 13, 23:00',
   'vessel_name': 'MDS ATHENA',
   'vessel_type': 'Bulk Carrier',
   'imo': '9450674',
   'flag': 'Marshall Islands',
   'built': 2025,
   'gt': 24464,
   'dwt': 40515,
   'size': '183 x 32'},
  {'time': 'Mar 14, 01:00',
   'vessel_name': 'WAN HAI 512',
   'vessel_type': 'Container Ship',
   'imo': '9457622',
   'flag': 'Singapore',
   'built': 2025,
   'gt': 24464,
   'dwt': 40515,
   'size': '183 x 32'},
  {'time': 'Mar 14, 14:00',
   'vessel_name': 'LORI',
   'vessel_type': 'Container Ship',
   'imo': '9631125',
   'flag': 'Liberi

In [25]:
parse_vessel_table(soup, "in-port")

[{'time': 'Mar 10, 12:52',
  'vessel_name': 'MAERSK SALINA',
  'vessel_type': 'Container Ship',
  'imo': '9352030',
  'flag': 'Hong Kong',
  'built': 2015,
  'gt': 12136,
  'dwt': 19999,
  'size': '144 x 24'},
 {'time': 'Mar 10, 13:28',
  'vessel_name': 'MSC MELISSA',
  'vessel_type': 'Container Ship',
  'imo': '9226918',
  'flag': 'Panama',
  'built': 2015,
  'gt': 12136,
  'dwt': 19999,
  'size': '144 x 24'},
 {'time': 'Mar 10, 13:24',
  'vessel_name': 'NEW FACE',
  'vessel_type': 'Bulk Carrier',
  'imo': '9727443',
  'flag': 'Panama',
  'built': 2015,
  'gt': 12136,
  'dwt': 19999,
  'size': '144 x 24'},
 {'time': 'Mar 10, 13:26',
  'vessel_name': 'CS CELESTE',
  'vessel_type': 'Bulk Carrier',
  'imo': '9670418',
  'flag': 'Marshall Islands',
  'built': 2015,
  'gt': 12136,
  'dwt': 19999,
  'size': '144 x 24'},
 {'time': 'Mar 10, 13:25',
  'vessel_name': 'CENTURION GORYO',
  'vessel_type': 'Bulk Carrier',
  'imo': '9965540',
  'flag': 'Liberia',
  'built': 2015,
  'gt': 12136,
  'd

In [39]:
parse_vessel_table(soup, "arrivals")

[{'time': 'Feb 19, 08:32',
  'vessel_name': 'GULLHOLMEN ISLAND',
  'vessel_type': 'Bulk Carrier',
  'imo': '9605085',
  'flag': 'Hong Kong',
  'built': 2016,
  'gt': 268,
  'dwt': 106,
  'size': '25 x 11'},
 {'time': 'Feb 19, 06:49',
  'vessel_name': 'TITAN',
  'vessel_type': 'Tug',
  'imo': '1039993',
  'flag': 'Chile',
  'built': 2016,
  'gt': 268,
  'dwt': 106,
  'size': '25 x 11'},
 {'time': 'Feb 19, 06:48',
  'vessel_name': 'RINIHUE',
  'vessel_type': 'Tug',
  'imo': '9740691',
  'flag': 'Chile',
  'built': 2016,
  'gt': 268,
  'dwt': 106,
  'size': '25 x 11'},
 {'time': 'Feb 19, 06:48',
  'vessel_name': 'MORNING PILOT',
  'vessel_type': 'Vehicles Carrier',
  'imo': '9669031',
  'flag': 'Marshall Islands',
  'built': 2016,
  'gt': 268,
  'dwt': 106,
  'size': '25 x 11'},
 {'time': 'Feb 19, 06:37',
  'vessel_name': 'TAYLOR ELQUI',
  'vessel_type': 'Pilot',
  'imo': '725002161',
  'flag': 'Chile',
  'built': 2016,
  'gt': 268,
  'dwt': 106,
  'size': '25 x 11'},
 {'time': 'Feb 19, 0

In [40]:
parse_vessel_table(soup, "departures")

[{'time': 'Feb 19, 08:17',
  'vessel_name': 'GULLHOLMEN ISLAND',
  'vessel_type': 'Bulk Carrier',
  'imo': '9605085',
  'flag': 'Hong Kong',
  'built': 2016,
  'gt': 268,
  'dwt': 106,
  'size': '25 x 11'},
 {'time': 'Feb 19, 06:32',
  'vessel_name': 'TITAN',
  'vessel_type': 'Tug',
  'imo': '1039993',
  'flag': 'Chile',
  'built': 2016,
  'gt': 268,
  'dwt': 106,
  'size': '25 x 11'},
 {'time': 'Feb 19, 06:29',
  'vessel_name': 'RINIHUE',
  'vessel_type': 'Tug',
  'imo': '9740691',
  'flag': 'Chile',
  'built': 2016,
  'gt': 268,
  'dwt': 106,
  'size': '25 x 11'},
 {'time': 'Feb 19, 06:24',
  'vessel_name': 'TAYLOR ELQUI',
  'vessel_type': 'Pilot',
  'imo': '725002161',
  'flag': 'Chile',
  'built': 2016,
  'gt': 268,
  'dwt': 106,
  'size': '25 x 11'},
 {'time': 'Feb 18, 21:17',
  'vessel_name': 'RINIHUE',
  'vessel_type': 'Tug',
  'imo': '9740691',
  'flag': 'Chile',
  'built': 2016,
  'gt': 268,
  'dwt': 106,
  'size': '25 x 11'},
 {'time': 'Feb 18, 21:16',
  'vessel_name': 'TORDO

In [26]:
data = {
    "source": SOURCE_NAME,
    "port_code": port_code.replace("001", ""),  # CLSAI001 -> CLSAI
    "port_name": port_name,
    "scrape_timestamp": datetime.utcnow().isoformat() + "Z",
    "summary": parse_port_summary(soup),
    "expected_vessels": parse_vessel_table(soup, "expected"),
    "vessels_in_port": parse_vessel_table(soup, "in-port"),
    "recent_arrivals": parse_vessel_table(soup, "arrivals"),
    "recent_departures": parse_vessel_table(soup, "departures"),
}

In [27]:
data

{'source': 'vesselfinder',
 'port_code': 'CLSAI',
 'port_name': 'San Antonio',
 'scrape_timestamp': '2026-03-10T13:34:02.426931Z',
 'summary': {'expected_ships': 11, 'ships_in_port': 18},
 'expected_vessels': [{'time': 'Mar 11, 00:00',
   'vessel_name': 'MSC EDNA',
   'vessel_type': 'Container Ship',
   'imo': '1016680',
   'flag': 'Liberia',
   'built': 2025,
   'gt': 24464,
   'dwt': 40515,
   'size': '183 x 32'},
  {'time': 'Mar 12, 14:05',
   'vessel_name': 'ASL SAKURA',
   'vessel_type': 'Bulk Carrier',
   'imo': '9961922',
   'flag': 'Marshall Islands',
   'built': 2025,
   'gt': 24464,
   'dwt': 40515,
   'size': '183 x 32'},
  {'time': 'Mar 13, 23:00',
   'vessel_name': 'MDS ATHENA',
   'vessel_type': 'Bulk Carrier',
   'imo': '9450674',
   'flag': 'Marshall Islands',
   'built': 2025,
   'gt': 24464,
   'dwt': 40515,
   'size': '183 x 32'},
  {'time': 'Mar 14, 01:00',
   'vessel_name': 'WAN HAI 512',
   'vessel_type': 'Container Ship',
   'imo': '9457622',
   'flag': 'Singapor

In [ ]:
try:
    scraper = VesselFinderPortScraper()
    data = scraper.scrape_all_ports()
    
    if data:
        save_data(data)
        print_summary(data)
        logger.info("Scraping completed successfully")
    else:
        logger.warning("No data collected")
        sys.exit(1)

except Exception as e:
    logger.error(f"Scraping failed: {e}")
    sys.exit(1)